In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV 
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
data = pd.read_csv('cpi_data.csv',parse_dates=['Date'])

In [3]:
data = data.set_index('Date')

In [4]:
categories = data.columns
models = {}
results = {}

In [5]:
date = '2026-01-01'

In [ ]:
for category in categories:
    df = data[[category]].copy()
    df = np.log(df)
    
    df['lag_1'] = df[category].shift(1)          
    df['lag_12'] = df[category].shift(12)        
    df['diff_1'] = df[category].shift(1) - df[category].shift(2)   
    df['month_sin'] = np.sin(2 * np.pi * df.index.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df.index.month / 12)
    df.dropna(inplace=True)

    x_train = df.drop(category,axis = 1).loc[:date]
    y_train = df[category].loc[:date]
    x_test = df.drop(category,axis = 1).loc[date:]
    y_test = df[category].loc[date:]

    model = RidgeCV(alphas=[0.1, 1.0, 10.0])
    model.fit(x_train, y_train)
    models[category] = model

    predictions = np.exp(model.predict(x_test))
    mape = mean_absolute_percentage_error(np.exp(y_test), predictions) * 100
    results[category] = mape
    print(f"Mean Absolute Percentage Error for {category}: {mape:.2f}% \n")

    

Mean Absolute Percentage Error for Education: 1.49% 

Mean Absolute Percentage Error for Energy: 3.95% 

Mean Absolute Percentage Error for Food: 0.18% 

Mean Absolute Percentage Error for Housing: 0.09% 

Mean Absolute Percentage Error for Medical: 1.76% 

Mean Absolute Percentage Error for Transportation: 1.94% 



In [ ]:
print("Food Model Coefficients:")
print(models['Food'].coef_)

print("\nEnergy Model Coefficients:")
print(models['Energy'].coef_)

Food Model Coefficients:
[ 6.23246680e-01  3.50189425e-01  2.29161287e-02 -1.18635161e-04
 -1.56679822e-05]

Energy Model Coefficients:
[ 0.91142205  0.03606185  0.159472    0.00978477 -0.00700298]


In [8]:
log_data = np.log(data)
log_data = log_data.dropna()

In [ ]:
def forecast_future(data, models, category, n_steps=6):
    log_series = data[category].copy()
    last_date = log_series.index[-1]
    freq = pd.infer_freq(log_series.index) or 'MS'
    future_dates = pd.date_range(start=last_date, periods=n_steps + 1, freq=freq)[1:]

    history = log_series.copy()   
    preds = []

    for date in future_dates:
        lag_1 = history.iloc[-1]
        lag_12 = history.iloc[-12]
        diff_1 = history.iloc[-1] - history.iloc[-2]
        month_sin = np.sin(2 * np.pi * date.month / 12)
        month_cos = np.cos(2 * np.pi * date.month / 12)

        x_new = pd.DataFrame(
            [[lag_1, lag_12, diff_1, month_sin, month_cos]],
            columns=['lag_1', 'lag_12', 'diff_1', 'month_sin', 'month_cos'],
            index=[date]
        )
        

        pred_log = models[category].predict(x_new)[0]
        history.loc[date] = pred_log   
        preds.append(np.exp(pred_log))

    return pd.Series(preds, index=future_dates, name=category)

In [10]:
forecasts = {}
for category in categories:
    forecasts[category] = forecast_future(log_data, models, category, n_steps=6)

In [11]:
forecast_df = pd.DataFrame(forecasts)
forecast_df.head()

,Education,Energy,Food,Housing,Medical,Transportation
2026-06-01,145.159815,343.665432,349.083195,358.679826,584.304254,296.005058
2026-07-01,144.687371,336.763594,349.448716,359.531064,581.272459,292.150077
2026-08-01,144.694307,327.829582,349.891072,360.246786,581.062381,287.840169
2026-09-01,144.856860,318.016898,350.625114,361.080478,580.970103,283.514358
2026-10-01,144.956938,308.493697,351.481169,361.744996,581.205201,279.610181


In [12]:
forecast_df.to_csv('regression_predictions.csv')